# Flight Duration — Machine Learning

Predicting `flight_duration_min` from airport and distance features using scikit-learn.

## Setup — Load data

In [1]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import config

engine = create_engine(
    f"postgresql+psycopg2://{config.DB_USER}:{config.DB_PASSWORD}@{config.DB_HOST}:{config.DB_PORT}/{config.DB_NAME}"
)

df = pd.read_sql("SELECT * FROM flights_clean", engine)
print(f"Loaded {len(df)} rows")

TARGET = "flight_duration_min"
df.head()

Loaded 988 rows


,icao24,callsign,estdepartureairport,estarrivalairport,first_seen,last_seen,flight_duration_min,dep_horiz_dist,arr_horiz_dist
0,008081,LNK861C,FAOR,FALL,2026-08-08 08:18:51+00:00,2026-08-08 08:38:14+00:00,19.383333,3784.0,11552.0
1,300ab7,IHMBS,LFLJ,LFHM,2026-08-08 07:41:18+00:00,2026-08-08 08:10:46+00:00,29.466667,94694.0,17967.0
2,343104,ECISV,LEAP,LEAP,2026-08-08 07:34:22+00:00,2026-08-08 08:10:53+00:00,36.516667,7801.0,5199.0
3,343104,ECISV,LEAP,LEAP,2026-08-08 08:21:00+00:00,2026-08-08 08:34:14+00:00,13.233333,3098.0,4556.0
4,344218,ANE2305,LEVX,LELN,2026-08-08 07:58:32+00:00,2026-08-08 08:28:42+00:00,30.166667,13297.0,137490.0


## Feature engineering

- Drop rows where the target is null.
- Group airports appearing fewer than 20 times into `OTHER`, then one-hot encode.
- Use `dep_horiz_dist` and `arr_horiz_dist` as numeric features.
- Drop remaining rows with nulls in the feature columns.

In [2]:
# Column names come back lowercased from PostgreSQL.
DEP_COL = "estdepartureairport"
ARR_COL = "estarrivalairport"
NUMERIC_COLS = ["dep_horiz_dist", "arr_horiz_dist"]

# Drop rows where the target is null.
df = df.dropna(subset=[TARGET])

# Replace rare airports (fewer than 20 occurrences) with 'OTHER'.
MIN_COUNT = 20
for col in [DEP_COL, ARR_COL]:
    counts = df[col].value_counts()
    frequent = counts[counts >= MIN_COUNT].index
    df[col] = df[col].where(df[col].isin(frequent), "OTHER")

# One-hot encode the airport columns.
encoded = pd.get_dummies(df[[DEP_COL, ARR_COL]], columns=[DEP_COL, ARR_COL])

# Assemble the feature matrix.
X = pd.concat([df[NUMERIC_COLS], encoded], axis=1)
y = df[TARGET]

# Drop rows with any nulls left in the feature columns.
mask = X.notnull().all(axis=1)
X = X[mask]
y = y[mask]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (988, 15)
Target shape: (988,)


## Train / test split (80 / 20)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

Train rows: 790
Test rows: 198


## Model 1 — Linear Regression

In [4]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

print("Linear Regression")
print("MAE :", lr_mae)
print("RMSE:", lr_rmse)
print("R2  :", lr_r2)

Linear Regression
MAE : 67.78347157843406
RMSE: 105.19053450246044
R2  : 0.019419603732559465


## Model 2 — Random Forest Regressor

In [5]:
rf = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest (max_depth=6)")
print("MAE :", rf_mae)
print("RMSE:", rf_rmse)
print("R2  :", rf_r2)

Random Forest (max_depth=6)
MAE : 64.99240049595728
RMSE: 103.55012212732683
R2  : 0.04976480099464309


## Overfitting check — Random Forest at different depths

Compare train R² and test R² for `max_depth` of 2, 6, and None.

In [6]:
results = []
for depth in [2, 6, None]:
    model = RandomForestRegressor(
        n_estimators=100, max_depth=depth, random_state=42
    )
    model.fit(X_train, y_train)
    train_r2 = r2_score(y_train, model.predict(X_train))
    test_r2 = r2_score(y_test, model.predict(X_test))
    results.append({
        "max_depth": str(depth),
        "train_R2": round(train_r2, 4),
        "test_R2": round(test_r2, 4),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

max_depth  train_R2  test_R2
        2    0.0703   0.0509
        6    0.4073   0.0498
     None    0.8504  -0.1130


## Conclusion

_Adjust these statements to match the actual scores you see after running the notebook._

The Random Forest generally outperforms Linear Regression here, since flight duration depends on airport pairings in a non-linear way that a straight-line model cannot fully capture. In plain English, the R² tells us what fraction of the variation in flight duration the model explains (1.0 is perfect, 0 is no better than always guessing the average), while MAE is the average error in minutes and RMSE is a similar error measure that punishes large mistakes more heavily. Looking at the overfitting table, `max_depth=None` usually shows a very high train R² but a lower test R², which is the classic sign of overfitting — memorizing the training data instead of learning general patterns. `max_depth=2` tends to underfit, with both scores low because the trees are too shallow to capture the structure. `max_depth=6` typically gives the best balance, with train and test R² close together, so that is the depth I would choose. Overall the model is useful for rough flight-duration estimates, but the error in minutes should be kept in mind before relying on it for precise predictions.